# 04 - Puede un modelo aprender patrones de resistencia, en vez de buscarlos en un diccionario?

El pipeline de `03_pipeline_organizado.ipynb` detecta genes de resistencia (ARG) comparando cada ORF
predicho contra la base curada de CARD: RGI hace, en esencia, una busqueda por alineamiento/homologia
(BLASTP + umbrales de bit-score curados). Es confiable, pero solo encuentra lo que ya esta catalogado.

Este cuaderno prueba una idea distinta: **entrenar un modelo con las secuencias de referencia de CARD
(no con los hits que ya encontro RGI en nuestras muestras)**, para ver si aprende patrones de secuencia
(composicion de k-mers) asociados a resistencia, y luego evaluar ese modelo sobre las secuencias reales
de nuestras muestras -- incluyendo ORFs que RGI nunca marco -- para ver si generaliza mas alla de la
lista de genes conocidos.

Puntos de diseno importantes (se detallan en cada seccion):

- **Entrenamos con CARD, evaluamos con lo real.** Los ARG que RGI ya encontro en nuestras muestras se
  apartan como conjunto de evaluacion "del mundo real" y nunca se usan para entrenar.
- **CARD son genes completos; nuestros ORFs reales son fragmentos** de un ensamblaje metagenomico
  submuestreado. Si no corregimos esto, el modelo podria aprender a distinguir "secuencia completa" vs.
  "fragmento" en lugar de un patron real de resistencia -- por eso fragmentamos las secuencias de CARD a
  una distribucion de longitud similar a la real antes de entrenar.
- **La prueba de generalizacion real es por familia de gen, no por secuencia individual.** Separamos
  familias completas de CARD (p. ej. todas las variantes de `vanW`) entre entrenamiento y prueba, para
  medir si el modelo reconoce algo genuinamente nuevo y no solo memoriza secuencias parecidas.
- Todo el cuaderno usa ML clasico (k-mers + XGBoost / regresion logistica), ya anticipado en
  `environment.yml` (`xgboost`, `imbalanced-learn`, `kmc`). No se necesita GPU y el entrenamiento en si
  toma minutos en un portatil; el unico paso potencialmente lento es la verificacion opcional por BLAST
  remoto al final.

Requiere haber corrido `03_pipeline_organizado.ipynb` sobre al menos algunas muestras (usa
`localDB/card.json`, `work/<run>/genes.fna` y `results/<run>/rgi_<run>.txt`).

## 1. Imports y configuracion

In [ ]:
import json
import random
import re
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from Bio import SeqIO
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

random.seed(42)
np.random.seed(42)

DIR_WORK = Path("work")
DIR_RESULTS = Path("results")
CARD_PATH = Path("localDB/card.json")
K = 4  # tamano de k-mer (4^4 = 256 dimensiones; k=5 o 6 quedan como posible mejora, ver seccion 8)

## 2. Conjunto positivo: secuencias de referencia de CARD

Parseamos `localDB/card.json` (la misma base que usa `ensure_card_db()` en el cuaderno 03) y extraemos,
por cada modelo, su secuencia de ADN de referencia junto con la familia de gen, clase de droga y
mecanismo de resistencia segun `ARO_category`. Estas ~6400 secuencias son el **unico** origen de
etiquetas positivas para entrenar -- deliberadamente no usamos los hits que RGI ya encontro en nuestras
muestras.

In [2]:
def cargar_card_positivos(card_path=CARD_PATH):
    data = json.loads(card_path.read_text())
    filas = []
    for model_id, modelo in data.items():
        if not isinstance(modelo, dict):
            continue  # claves de metadata como _version, _timestamp
        secuencias = modelo.get("model_sequences", {}).get("sequence", {})
        categorias = modelo.get("ARO_category", {})
        familia = clase_droga = mecanismo = None
        for cat in categorias.values():
            clase = cat.get("category_aro_class_name")
            if clase == "AMR Gene Family" and familia is None:
                familia = cat.get("category_aro_name")
            elif clase == "Drug Class" and clase_droga is None:
                clase_droga = cat.get("category_aro_name")
            elif clase == "Resistance Mechanism" and mecanismo is None:
                mecanismo = cat.get("category_aro_name")
        for seq in secuencias.values():
            dna = seq.get("dna_sequence", {}).get("sequence")
            if not dna:
                continue
            filas.append({
                "model_id": model_id,
                "aro_name": modelo.get("ARO_name"),
                "familia": familia or "desconocida",
                "clase_droga": clase_droga or "desconocida",
                "mecanismo": mecanismo or "desconocido",
                "secuencia": dna.upper(),
            })
    return pd.DataFrame(filas)


card_df = cargar_card_positivos()
print(f"Secuencias positivas de CARD: {len(card_df)}")
print(f"Familias de genes distintas: {card_df['familia'].nunique()}")
card_df.head()

Secuencias positivas de CARD: 6404
Familias de genes distintas: 470


,model_id,aro_name,familia,clase_droga,mecanismo,secuencia
0,2,CblA-1,CblA beta-lactamase,cephalosporin,antibiotic inactivation,ATGAAAGCATATTTCATCGCCATACTTACCTTATTCACTTGTATAG...
1,4,SHV-52,SHV beta-lactamase,cephalosporin,antibiotic inactivation,ATGCGTTATATTCGCCTGTGTATTATCTCCCTGTTAGCCGCCCTGC...
2,5,dfrF,trimethoprim resistant dihydrofolate reductase...,diaminopyrimidine antibiotic,antibiotic target replacement,ATGATAGGTTTGATTGTTGCGAGGTCAAAGAATAATGTTATAGGCA...
3,7,CTX-M-130,CTX-M beta-lactamase,cephalosporin,antibiotic inactivation,ATGGTGACAAAGAGAGTGCAACGGATGATGTTCGCGGCGGCGGCGT...
4,8,NDM-6,NDM beta-lactamase,carbapenem,antibiotic inactivation,ATGGAATTGCCCAATATTATGCACCCGGTCGCGAAGCTGAGCACCG...


## 3. ORFs reales: separar hits ya conocidos (evaluacion) del resto (pool negativo)

Para cada muestra procesada por el cuaderno 03, `work/<run>/genes.fna` contiene **todos** los ORFs que
predijo Prodigal, y `results/<run>/rgi_<run>.txt` contiene el subconjunto que RGI marco como ARG (el
encabezado FASTA es identico al valor de `ORF_ID` en la tabla de RGI, asi que separarlos es una simple
diferencia de conjuntos).

- Los hits de RGI se apartan como **conjunto de evaluacion del mundo real** (`positivos_reales`): nunca
  se usan para entrenar, solo para ver si el modelo los recupera a partir del patron de secuencia.
- Todo lo demas (~99.8% de los ORFs) es el **pool negativo** de entrenamiento: en una muestra
  metagenomica real, la enorme mayoria de genes no son de resistencia.

In [3]:
def cargar_orfs_muestra(run):
    fna = DIR_WORK / run / "genes.fna"
    rgi_txt = DIR_RESULTS / run / f"rgi_{run}.txt"
    if not fna.exists():
        return pd.DataFrame()
    hits_rgi = set()
    if rgi_txt.exists():
        hits_rgi = set(pd.read_csv(rgi_txt, sep="\t")["ORF_ID"])
    filas = [
        {
            "run": run,
            "orf_id": rec.description,
            "secuencia": str(rec.seq).upper(),
            "es_hit_rgi": rec.description in hits_rgi,
        }
        for rec in SeqIO.parse(fna, "fasta")
    ]
    return pd.DataFrame(filas)


runs = sorted(p.name for p in DIR_WORK.iterdir() if (p / "genes.fna").exists())
orfs_df = pd.concat([cargar_orfs_muestra(r) for r in runs], ignore_index=True)

positivos_reales = orfs_df[orfs_df["es_hit_rgi"]].reset_index(drop=True)
pool_negativo = orfs_df[~orfs_df["es_hit_rgi"]].reset_index(drop=True)

print(f"ORFs totales: {len(orfs_df)} en {len(runs)} muestras")
print(f"  Hits RGI (evaluacion real, NO se entrena con ellos): {len(positivos_reales)}")
print(f"  Pool negativo para entrenamiento: {len(pool_negativo)}")

ORFs totales: 377345 en 34 muestras
  Hits RGI (evaluacion real, NO se entrena con ellos): 489
  Pool negativo para entrenamiento: 376856


## 4. Fragmentar CARD para igualar la distribucion de longitud real

Las secuencias de referencia de CARD son genes completos (mediana ~876 pb); nuestros ORFs reales son
fragmentos de un ensamblaje submuestreado (mediana bastante menor, y muchos truncados en el borde de un
contig). Entrenar solo con genes completos de CARD dejaria una senal facil de explotar: "esta secuencia
es larga y completa" en vez de un patron de composicion real. Por eso generamos, ademas de la secuencia
completa, varios recortes aleatorios de cada secuencia de CARD con longitudes muestreadas de la
distribucion real observada en `pool_negativo`.

In [4]:
longitudes_reales = pool_negativo["secuencia"].str.len()
print(longitudes_reales.describe())


def fragmentar(secuencia, longitud, rng):
    n = len(secuencia)
    if longitud >= n:
        return secuencia
    inicio = rng.randrange(0, n - longitud + 1)
    return secuencia[inicio:inicio + longitud]


def construir_positivos_entrenamiento(card_df, longitudes_objetivo, fragmentos_por_secuencia=3, semilla=42):
    rng = random.Random(semilla)
    longitudes_objetivo = longitudes_objetivo.to_numpy()
    filas = []
    for _, fila in card_df.iterrows():
        filas.append(fila.to_dict())  # version completa: al modelo tambien le sirve ver el gen entero
        for _ in range(fragmentos_por_secuencia):
            longitud = int(rng.choice(longitudes_objetivo))
            frag = fragmentar(fila["secuencia"], longitud, rng)
            if len(frag) < 20:
                continue
            nueva = fila.to_dict()
            nueva["secuencia"] = frag
            filas.append(nueva)
    return pd.DataFrame(filas)


card_entrenamiento = construir_positivos_entrenamiento(card_df, longitudes_reales)
print(f"Positivos de entrenamiento tras fragmentar: {len(card_entrenamiento)} "
      f"(de {len(card_df)} secuencias originales de CARD)")

count    376856.000000
mean        430.237284
std         322.245037
min          60.000000
25%         234.000000
50%         360.000000
75%         519.000000
max       16731.000000
Name: secuencia, dtype: float64


Positivos de entrenamiento tras fragmentar: 25616 (de 6404 secuencias originales de CARD)


## 5. Features: frecuencia de k-mers de nucleotidos

Usamos composicion de k-mers (k=4 por defecto, "firma genomica" tetranucleotidica) en vez de alineamiento
contra una referencia: es una tecnica reference-free establecida (usada p. ej. para binning taxonomico) y
es exactamente el tipo de senal "de patron" que le falta a una busqueda por diccionario. Se calcula
directamente en Python (para secuencias de este tamano no hace falta el binario `kmc` del entorno).

In [5]:
def vocabulario_kmers(k):
    return ["".join(p) for p in product("ACGT", repeat=k)]


def vector_kmer(secuencia, k, vocab_index):
    vec = np.zeros(len(vocab_index), dtype=np.float32)
    secuencia = re.sub(r"[^ACGT]", "", secuencia)
    total = 0
    for i in range(len(secuencia) - k + 1):
        idx = vocab_index.get(secuencia[i:i + k])
        if idx is not None:
            vec[idx] += 1
            total += 1
    if total:
        vec /= total
    return vec


def matriz_kmers(secuencias, k=K):
    vocab_index = {kmer: i for i, kmer in enumerate(vocabulario_kmers(k))}
    return np.vstack([vector_kmer(s, k, vocab_index) for s in secuencias])


negativos_entrenamiento = pool_negativo.sample(
    n=min(len(pool_negativo), len(card_entrenamiento) * 3), random_state=42
)

X_pos = matriz_kmers(card_entrenamiento["secuencia"])
X_neg = matriz_kmers(negativos_entrenamiento["secuencia"])

X = np.vstack([X_pos, X_neg])
y = np.concatenate([np.ones(len(X_pos)), np.zeros(len(X_neg))])
familias = np.concatenate([
    card_entrenamiento["familia"].to_numpy(),
    np.array(["(negativo)"] * len(X_neg)),
])

print(f"Dataset de entrenamiento: {X.shape}, positivos={int(y.sum())}, negativos={int((y == 0).sum())}")

Dataset de entrenamiento: (102464, 256), positivos=25616, negativos=76848


## 6. Dos particiones de evaluacion

- **Split aleatorio**: reparte secuencias al azar entre train/test. Sirve como chequeo de cordura, pero
  no responde la pregunta de generalizacion (secuencias muy parecidas de la misma familia pueden quedar
  en ambos lados).
- **Split por familia de gen (holdout)**: aparta ~20% de las 522 familias de CARD *completas* para
  prueba -- ninguna variante de esas familias se ve en entrenamiento. Esta es la prueba real de si el
  modelo generaliza mas alla de una lista conocida, en vez de solo memorizar.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

familias_unicas = sorted(set(card_entrenamiento["familia"]))
rng_split = random.Random(7)
rng_split.shuffle(familias_unicas)
n_holdout = max(1, int(0.2 * len(familias_unicas)))
familias_holdout = set(familias_unicas[:n_holdout])

es_holdout = np.isin(familias, list(familias_holdout))
es_positivo = y == 1

test_mask_fam = es_holdout & es_positivo
train_mask_fam = ~test_mask_fam

neg_idx = np.where(y == 0)[0]
neg_test_idx = rng_split.sample(list(neg_idx), k=min(len(neg_idx), int(0.2 * len(neg_idx))))
test_mask_fam[neg_test_idx] = True
train_mask_fam[neg_test_idx] = False

X_train_fam, y_train_fam = X[train_mask_fam], y[train_mask_fam]
X_test_fam, y_test_fam = X[test_mask_fam], y[test_mask_fam]

assert not (set(familias[train_mask_fam & es_positivo]) & familias_holdout), "fuga de familias al train"
print(f"Familias en holdout: {len(familias_holdout)}/{len(familias_unicas)}")
print(f"Split por familia -> train: {X_train_fam.shape}, test: {X_test_fam.shape}")

Familias en holdout: 94/470
Split por familia -> train: (83307, 256), test: (19157, 256)


## 7. Búsqueda de hiperparámetros (M, learning rate, profundidad)

Los tres hiperparámetros que más afectan el desempeño de XGBoost son el número de árboles ($M$), la
tasa de aprendizaje ($\eta$) y la profundidad de cada árbol. Antes de entrenar el modelo "oficial" de
la siguiente sección, exploramos una grilla pequeña de `learning_rate` y `max_depth`, dejando que
**early stopping** (contra la partición por familia, la que realmente mide generalización) elija $M$
automáticamente para cada combinación en vez de fijarlo de antemano.

Además del AUPRC de validación, se reporta la brecha entre AUPRC de entrenamiento y de validación como
señal de sobreajuste: una brecha grande indica que el modelo está memorizando el conjunto de
entrenamiento en vez de generalizar a familias nuevas.

In [7]:
from itertools import product as producto_grilla


def buscar_hiperparametros(X_train, y_train, X_val, y_val,
                            learning_rates=(0.05, 0.1, 0.2),
                            profundidades=(3, 5, 7),
                            n_estimators_max=1000,
                            paciencia=20):
    """Grilla learning_rate x max_depth; M se elige por early stopping contra (X_val, y_val)."""
    resultados = []
    for lr, profundidad in producto_grilla(learning_rates, profundidades):
        modelo = XGBClassifier(
            n_estimators=n_estimators_max,
            max_depth=profundidad,
            learning_rate=lr,
            scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
            eval_metric="aucpr",
            early_stopping_rounds=paciencia,
            random_state=42,
        )
        modelo.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train), (X_val, y_val)],
            verbose=False,
        )

        proba_train = modelo.predict_proba(X_train)[:, 1]
        proba_val = modelo.predict_proba(X_val)[:, 1]

        resultados.append({
            "learning_rate": lr,
            "max_depth": profundidad,
            "M_elegido": modelo.best_iteration + 1,
            "auprc_train": average_precision_score(y_train, proba_train),
            "auprc_val": average_precision_score(y_val, proba_val),
            "rocauc_val": roc_auc_score(y_val, proba_val),
        })

    df = pd.DataFrame(resultados)
    df["brecha_train_val"] = df["auprc_train"] - df["auprc_val"]
    return df.sort_values("auprc_val", ascending=False).reset_index(drop=True)


resultados_grid = buscar_hiperparametros(X_train_fam, y_train_fam, X_test_fam, y_test_fam)

pd.set_option("display.precision", 3)
display(resultados_grid)

mejor = resultados_grid.iloc[0]
LR_ELEGIDO = float(mejor["learning_rate"])
DEPTH_ELEGIDO = int(mejor["max_depth"])
print(f"\nMejor combinacion por AUPRC de validacion: learning_rate={LR_ELEGIDO}, "
      f"max_depth={DEPTH_ELEGIDO}, M={int(mejor['M_elegido'])} -> "
      f"AUPRC_val={mejor['auprc_val']:.3f} (brecha train-val={mejor['brecha_train_val']:.3f})\n\n"
      "Nota metodologica: esta busqueda usa (X_test_fam, y_test_fam) tanto para elegir M (early "
      "stopping) como para reportar el AUPRC de cada combinacion, lo que introduce un sesgo "
      "optimista leve en estos numeros al comparar hiperparametros entre si. Por eso, en el "
      "entrenamiento oficial de la seccion 7.1, M se elige con un 10% separado del conjunto de "
      "*entrenamiento* (nunca con el conjunto de prueba), y aqui solo adoptamos learning_rate y "
      "max_depth de la mejor combinacion, no su M exacto.")

,learning_rate,max_depth,M_elegido,auprc_train,auprc_val,rocauc_val,brecha_train_val
0,0.20,7,551,1.000,0.859,0.937,0.141
1,0.05,7,1000,1.000,0.857,0.936,0.143
2,0.10,7,486,1.000,0.854,0.934,0.146
3,0.05,5,1000,1.000,0.849,0.931,0.151
4,0.20,5,448,1.000,0.848,0.930,0.152
5,0.10,5,455,1.000,0.847,0.931,0.153
6,0.10,3,876,0.991,0.835,0.926,0.157
7,0.20,3,369,0.987,0.828,0.924,0.159
8,0.05,3,1000,0.981,0.823,0.921,0.158



Mejor combinacion por AUPRC de validacion: learning_rate=0.2, max_depth=7, M=551 -> AUPRC_val=0.859 (brecha train-val=0.141)

Nota metodologica: esta busqueda usa (X_test_fam, y_test_fam) tanto para elegir M (early stopping) como para reportar el AUPRC de cada combinacion, lo que introduce un sesgo optimista leve en estos numeros al comparar hiperparametros entre si. Por eso, en el entrenamiento oficial de la seccion 7.1, M se elige con un 10% separado del conjunto de *entrenamiento* (nunca con el conjunto de prueba), y aqui solo adoptamos learning_rate y max_depth de la mejor combinacion, no su M exacto.


## 7.1 Entrenamiento: regresión logística (baseline) y XGBoost

Con `learning_rate` y `max_depth` de la mejor combinación encontrada arriba, entrenamos y evaluamos
regresión logística (línea base) y XGBoost en las dos particiones de evaluación. A diferencia de la
búsqueda de la sección 7 -- que usaba el mismo conjunto de prueba tanto para elegir $M$ como para
reportar el AUPRC --, aquí $M$ se elige por early stopping contra un 10% separado del conjunto de
*entrenamiento*; el conjunto de prueba nunca participa en la elección de $M$, solo en el reporte
final.

In [8]:
def entrenar_evaluar(X_train, y_train, X_test, y_test, nombre_split,
                      xgb_learning_rate=LR_ELEGIDO, xgb_max_depth=DEPTH_ELEGIDO,
                      xgb_n_estimators_max=1000, xgb_paciencia=20):
    """Regresion logistica (baseline) y XGBoost. learning_rate/max_depth vienen de la busqueda de
    la seccion 7; M se elige por early stopping contra un 10% separado de (X_train, y_train) --
    (X_test, y_test) nunca se usa para elegir M, solo para reportar las metricas finales."""
    X_fit, X_es, y_fit, y_es = train_test_split(
        X_train, y_train, test_size=0.1, stratify=y_train, random_state=42
    )

    modelos = {
        "regresion logistica": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "xgboost": XGBClassifier(
            n_estimators=xgb_n_estimators_max, max_depth=xgb_max_depth, learning_rate=xgb_learning_rate,
            scale_pos_weight=(y_fit == 0).sum() / max((y_fit == 1).sum(), 1),
            eval_metric="aucpr", early_stopping_rounds=xgb_paciencia, random_state=42,
        ),
    }
    entrenados = {}
    for nombre_modelo, modelo in modelos.items():
        if nombre_modelo == "xgboost":
            modelo.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], verbose=False)
        else:
            modelo.fit(X_train, y_train)
        proba = modelo.predict_proba(X_test)[:, 1]
        pred = (proba >= 0.5).astype(int)
        print(f"\n=== {nombre_split} -- {nombre_modelo} ===")
        print(classification_report(y_test, pred, digits=3))
        extra_M = f"  (M={modelo.best_iteration + 1})" if nombre_modelo == "xgboost" else ""
        print(f"ROC-AUC: {roc_auc_score(y_test, proba):.3f}  "
              f"AUPRC: {average_precision_score(y_test, proba):.3f}{extra_M}")
        entrenados[nombre_modelo] = modelo
    return entrenados


modelos_random = entrenar_evaluar(X_train, y_train, X_test, y_test, "split aleatorio")
modelos_familia = entrenar_evaluar(X_train_fam, y_train_fam, X_test_fam, y_test_fam, "split por familia (holdout)")


=== split aleatorio -- regresion logistica ===
              precision    recall  f1-score   support

         0.0      0.933     0.879     0.905     15370
         1.0      0.691     0.811     0.746      5123

    accuracy                          0.862     20493
   macro avg      0.812     0.845     0.826     20493
weighted avg      0.873     0.862     0.866     20493

ROC-AUC: 0.921  AUPRC: 0.843



=== split aleatorio -- xgboost ===
              precision    recall  f1-score   support

         0.0      0.967     0.993     0.980     15370
         1.0      0.977     0.900     0.937      5123

    accuracy                          0.970     20493
   macro avg      0.972     0.946     0.958     20493
weighted avg      0.970     0.970     0.969     20493

ROC-AUC: 0.989  AUPRC: 0.980  (M=869)



=== split por familia (holdout) -- regresion logistica ===
              precision    recall  f1-score   support

         0.0      0.895     0.894     0.895     15369
         1.0      0.573     0.576     0.574      3788

    accuracy                          0.831     19157
   macro avg      0.734     0.735     0.735     19157
weighted avg      0.832     0.831     0.831     19157

ROC-AUC: 0.831  AUPRC: 0.628



=== split por familia (holdout) -- xgboost ===
              precision    recall  f1-score   support

         0.0      0.892     0.994     0.941     15369
         1.0      0.956     0.514     0.668      3788

    accuracy                          0.899     19157
   macro avg      0.924     0.754     0.805     19157
weighted avg      0.905     0.899     0.887     19157

ROC-AUC: 0.935  AUPRC: 0.854  (M=976)


## 8. Aplicar el modelo a las muestras reales y comparar contra RGI

Entrenamos un modelo final con **todas** las secuencias positivas construidas a partir de CARD (todas las
familias, no solo el 80% del holdout) para maximizar la senal de entrenamiento, y lo aplicamos sobre
**todos** los ORFs reales de nuestras 32 muestras (no solo los que RGI ya marco).

Usamos `learning_rate` y `max_depth` de la busqueda de la seccion 7. Como este modelo final no tiene
una particion de prueba propia (usa el 100% de los datos derivados de CARD), $M$ se elige en dos pasos:
(1) se aparta un 10% de (X, y) solo para early stopping y se entrena hasta encontrar el mejor $M$, y
(2) se descarta ese modelo y se reentrena desde cero con ese $M$ fijo sobre el 100% de los datos, para
no desperdiciar el 10% apartado en el modelo que finalmente se usa.

Dos numeros importan aqui:
- **Recall sobre los hits ya conocidos de RGI**: si el modelo no recupera la mayoria de estos, no aprendio
  nada util y no vale la pena mirar el resto.
- **Candidatos nuevos**: ORFs que el modelo marca como probable ARG pero que RGI *no* marco. Este es el
  resultado que realmente responde la pregunta original -- pero son candidatos sin verificar, no ARG
  confirmados (ver seccion 9).

In [9]:
X_fit_final, X_es_final, y_fit_final, y_es_final = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

modelo_seleccion_M = XGBClassifier(
    n_estimators=1000, max_depth=DEPTH_ELEGIDO, learning_rate=LR_ELEGIDO,
    scale_pos_weight=(y_fit_final == 0).sum() / (y_fit_final == 1).sum(),
    eval_metric="aucpr", early_stopping_rounds=20, random_state=42,
)
modelo_seleccion_M.fit(X_fit_final, y_fit_final, eval_set=[(X_es_final, y_es_final)], verbose=False)
M_FINAL = modelo_seleccion_M.best_iteration + 1
print(f"M elegido por early stopping para el modelo final: {M_FINAL}")

modelo_final = XGBClassifier(
    n_estimators=M_FINAL, max_depth=DEPTH_ELEGIDO, learning_rate=LR_ELEGIDO,
    scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
    eval_metric="logloss", random_state=42,
)
modelo_final.fit(X, y)

X_real = matriz_kmers(orfs_df["secuencia"])
orfs_df["proba_arg"] = modelo_final.predict_proba(X_real)[:, 1]

UMBRAL = 0.5
recall_conocidos = orfs_df.loc[orfs_df["es_hit_rgi"], "proba_arg"].ge(UMBRAL).mean()
print(f"Recall sobre los {orfs_df['es_hit_rgi'].sum()} ARG ya conocidos (RGI): {recall_conocidos:.1%}")

candidatos_nuevos = (
    orfs_df[(~orfs_df["es_hit_rgi"]) & (orfs_df["proba_arg"] >= UMBRAL)]
    .sort_values("proba_arg", ascending=False)
)
print(f"Candidatos NO catalogados por RGI, marcados como posible ARG por el modelo: {len(candidatos_nuevos)} "
      f"(de {len(pool_negativo)} ORFs no-ARG)")
candidatos_nuevos[["run", "orf_id", "proba_arg"]].head(20)

M elegido por early stopping para el modelo final: 992


Recall sobre los 489 ARG ya conocidos (RGI): 53.0%
Candidatos NO catalogados por RGI, marcados como posible ARG por el modelo: 2111 (de 376856 ORFs no-ARG)


,run,orf_id,proba_arg
337432,ERR1135451,k119_3676_1 # 1 # 327 # 1 # ID=4095_1;partial=...,1.0
312691,ERR1135438,k119_1138_1 # 1 # 135 # -1 # ID=1133_1;partial...,1.0
39018,ERR1135196,k99_4625_2 # 603 # 677 # -1 # ID=7752_2;partia...,1.0
374287,ERR1135463,k119_6691_2 # 220 # 351 # -1 # ID=7867_2;parti...,1.0
195513,ERR1135222,k99_3827_7 # 4242 # 4784 # 1 # ID=11941_7;part...,1.0
367150,ERR1135463,k119_9607_2 # 491 # 619 # -1 # ID=2317_2;parti...,1.0
301419,ERR1135436,k119_6574_1 # 2 # 391 # 1 # ID=7367_1;partial=...,1.0
23402,ERR1135194,k99_2010_1 # 3 # 743 # 1 # ID=8053_1;partial=1...,1.0
368048,ERR1135463,k119_9669_1 # 3 # 143 # -1 # ID=3044_1;partial...,1.0
33242,ERR1135196,k99_10202_1 # 399 # 479 # 1 # ID=3266_1;partia...,1.0


## 8.1 Efecto del tamaño de k-mer (k)

Hasta ahora usamos $k=4$ (256 dimensiones) sin haberlo comparado nunca contra otros valores. Aquí
variamos solo $k$ -- manteniendo `learning_rate` y `max_depth` de la sección 7 fijos, y reusando
exactamente la misma partición `train_mask_fam`/`test_mask_fam` (que no depende de $k$, solo de qué
fila corresponde a qué familia) -- para aislar el efecto de $k$ del resto de la búsqueda de
hiperparámetros.

Se espera un compromiso, no una mejora monótona: $k$ más grande captura motivos más específicos, pero
también vuelve más disperso (ruidoso) el vector de frecuencias de los fragmentos más cortos -- un
fragmento de 60pb solo tiene ~55 k-meros de longitud 6, repartidos entre 4096 casillas posibles.

In [ ]:
def evaluar_k(k, learning_rate=LR_ELEGIDO, max_depth=DEPTH_ELEGIDO,
              n_estimators_max=1000, paciencia=20):
    """Reusa exactamente la particion train_mask_fam/test_mask_fam (no depende de k), solo cambia
    la extraccion de features. learning_rate y max_depth quedan fijos en los de la seccion 7."""
    Xk_pos = matriz_kmers(card_entrenamiento["secuencia"], k=k)
    Xk_neg = matriz_kmers(negativos_entrenamiento["secuencia"], k=k)
    Xk = np.vstack([Xk_pos, Xk_neg])

    Xk_train_fam, yk_train_fam = Xk[train_mask_fam], y[train_mask_fam]
    Xk_test_fam, yk_test_fam = Xk[test_mask_fam], y[test_mask_fam]

    X_fit, X_es, y_fit, y_es = train_test_split(
        Xk_train_fam, yk_train_fam, test_size=0.1, stratify=yk_train_fam, random_state=42
    )
    modelo = XGBClassifier(
        n_estimators=n_estimators_max, max_depth=max_depth, learning_rate=learning_rate,
        scale_pos_weight=(y_fit == 0).sum() / max((y_fit == 1).sum(), 1),
        eval_metric="aucpr", early_stopping_rounds=paciencia, random_state=42,
    )
    modelo.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], verbose=False)

    proba = modelo.predict_proba(Xk_test_fam)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        "k": k,
        "dimensiones": Xk.shape[1],
        "M_elegido": modelo.best_iteration + 1,
        "auprc_val": average_precision_score(yk_test_fam, proba),
        "rocauc_val": roc_auc_score(yk_test_fam, proba),
        "recall_val": recall_score(yk_test_fam, pred),
        "precision_val": precision_score(yk_test_fam, pred),
    }


resultados_k = pd.DataFrame([evaluar_k(k) for k in (2, 3, 4, 5, 6)])
resultados_k = resultados_k.sort_values("auprc_val", ascending=False).reset_index(drop=True)
display(resultados_k)

MEJOR_K = int(resultados_k.iloc[0]["k"])
auprc_k4 = resultados_k.loc[resultados_k["k"] == 4, "auprc_val"].values[0]
print(f"\nMejor k por AUPRC de validacion: k={MEJOR_K} "
      f"(k=4, el usado hasta ahora, tiene AUPRC_val={auprc_k4:.3f})")

## 8.2 Fuente de las secuencias positivas de entrenamiento: CARD, ARG reales, o mezcla

Hasta ahora entrenamos siempre con las secuencias de referencia de CARD y evaluamos contra los ARG
reales confirmados por RGI. Aquí comparamos tres fuentes de secuencias positivas de entrenamiento,
manteniendo todo lo demás idéntico -- mismos negativos de entrenamiento, mismo `learning_rate`/
`max_depth` de la sección 7, y el mismo conjunto de prueba compartido (positivos de CARD en familias
no vistas + un 20% separado de los ARG reales, que nunca se usa para entrenar en ninguna
configuración):

- **CARD (actual)**: los positivos de entrenamiento de siempre (familias no-holdout de CARD).
- **RGI real (reverso)**: solo el 80% de los ARG reales confirmados -- la dirección opuesta del
  experimento original (entrenar con lo real, evaluar contra CARD).
- **Mezcla**: CARD + ese mismo 80% de ARG reales combinados, sin usar el 100% de ninguna de las dos
  poblaciones, para poder seguir evaluando contra un conjunto de prueba genuinamente separado de
  ambas fuentes.

El número que más importa aquí es `recall_arg_real_holdout`: el recall sobre ARG reales que nunca
participaron en el entrenamiento de esa configuración específica -- es la métrica más directamente
comparable con el 53.0% de la sección 8.

In [ ]:
secuencias_por_fila = pd.concat(
    [card_entrenamiento["secuencia"], negativos_entrenamiento["secuencia"]],
    ignore_index=True,
)

X_train_fam_pos = X_train_fam[y_train_fam == 1]
secuencias_card_train_pos = secuencias_por_fila[train_mask_fam & (y == 1)]
secuencias_card_train_neg = secuencias_por_fila[train_mask_fam & (y == 0)]
X_train_fam_neg = X_train_fam[y_train_fam == 0]

reales_shuffled = positivos_reales.sample(frac=1, random_state=42).reset_index(drop=True)
n_reales_test = max(1, round(0.2 * len(reales_shuffled)))
reales_test_df = reales_shuffled.iloc[:n_reales_test]
reales_train_df = reales_shuffled.iloc[n_reales_test:]

Xk_reales_train = matriz_kmers(reales_train_df["secuencia"])
Xk_reales_test = matriz_kmers(reales_test_df["secuencia"])

X_test_compartido = np.vstack([X_test_fam, Xk_reales_test])
y_test_compartido = np.concatenate([y_test_fam, np.ones(len(reales_test_df))])
origen_test = np.array(["card"] * len(y_test_fam) + ["rgi_real"] * len(reales_test_df))

print(f"ARG reales: {len(reales_train_df)} entrenamiento / {len(reales_test_df)} prueba (80/20)")
print(f"Conjunto de prueba compartido: {len(y_test_compartido)} filas "
      f"({(origen_test == 'card').sum()} de CARD-holdout, {(origen_test == 'rgi_real').sum()} ARG reales holdout)")


def entrenar_evaluar_fuente(X_pos_train, nombre,
                             learning_rate=LR_ELEGIDO, max_depth=DEPTH_ELEGIDO,
                             n_estimators_max=1000, paciencia=20):
    """Entrena variando solo el origen de los POSITIVOS de entrenamiento; los negativos
    (X_train_fam_neg) y el conjunto de prueba compartido son siempre los mismos, para que la
    comparacion entre fuentes sea limpia (una sola variable cambia por configuracion)."""
    X_train_cfg = np.vstack([X_pos_train, X_train_fam_neg])
    y_train_cfg = np.concatenate([np.ones(len(X_pos_train)), np.zeros(len(X_train_fam_neg))])

    X_fit, X_es, y_fit, y_es = train_test_split(
        X_train_cfg, y_train_cfg, test_size=0.1, stratify=y_train_cfg, random_state=42
    )
    modelo = XGBClassifier(
        n_estimators=n_estimators_max, max_depth=max_depth, learning_rate=learning_rate,
        scale_pos_weight=(y_fit == 0).sum() / max((y_fit == 1).sum(), 1),
        eval_metric="aucpr", early_stopping_rounds=paciencia, random_state=42,
    )
    modelo.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], verbose=False)

    proba = modelo.predict_proba(X_test_compartido)[:, 1]
    pred = (proba >= 0.5).astype(int)
    mask_card, mask_rgi = origen_test == "card", origen_test == "rgi_real"
    return {
        "fuente_positivos": nombre,
        "n_positivos_entrenamiento": len(X_pos_train),
        "M_elegido": modelo.best_iteration + 1,
        "auprc_total": average_precision_score(y_test_compartido, proba),
        "rocauc_total": roc_auc_score(y_test_compartido, proba),
        "recall_card_holdout": recall_score(y_test_compartido[mask_card], pred[mask_card]),
        "recall_arg_real_holdout": recall_score(y_test_compartido[mask_rgi], pred[mask_rgi]),
    }


configuraciones = {
    "CARD (actual)": X_train_fam_pos,
    "RGI real (reverso)": Xk_reales_train,
    "Mezcla (CARD + RGI real)": np.vstack([X_train_fam_pos, Xk_reales_train]),
}

resultados_fuente = pd.DataFrame([
    entrenar_evaluar_fuente(X_pos, nombre) for nombre, X_pos in configuraciones.items()
]).sort_values("recall_arg_real_holdout", ascending=False).reset_index(drop=True)
display(resultados_fuente)

MEJOR_FUENTE = resultados_fuente.iloc[0]["fuente_positivos"]
print(f"\nMejor fuente por recall en ARG reales holdout: {MEJOR_FUENTE}")

## 8.3 Recalculando M con un techo más alto para la mejor combinación

En la sección 8 (modelo final), $M$ se eligió por early stopping pero llegó a 992 de un techo de
1000 -- posible señal de que no había convergido del todo. Aquí tomamos la mejor combinación de $k$
(sección 8.1) y de fuente de positivos (sección 8.2), y repetimos la elección de $M$ con un techo
mucho más alto (3000) y más paciencia, para confirmar que el valor elegido realmente converge en vez
de truncarse artificialmente.

In [ ]:
def secuencias_para_fuente(nombre):
    if nombre == "CARD (actual)":
        return secuencias_card_train_pos
    elif nombre == "RGI real (reverso)":
        return reales_train_df["secuencia"]
    elif nombre == "Mezcla (CARD + RGI real)":
        return pd.concat([secuencias_card_train_pos, reales_train_df["secuencia"]], ignore_index=True)
    raise ValueError(nombre)


secs_pos_final = secuencias_para_fuente(MEJOR_FUENTE)
secs_neg_final = secuencias_card_train_neg  # mismos negativos que en 8.2, para consistencia

Xk_pos_final = matriz_kmers(secs_pos_final, k=MEJOR_K)
Xk_neg_final = matriz_kmers(secs_neg_final, k=MEJOR_K)
X_final_cfg = np.vstack([Xk_pos_final, Xk_neg_final])
y_final_cfg = np.concatenate([np.ones(len(Xk_pos_final)), np.zeros(len(Xk_neg_final))])

X_fit_final2, X_es_final2, y_fit_final2, y_es_final2 = train_test_split(
    X_final_cfg, y_final_cfg, test_size=0.1, stratify=y_final_cfg, random_state=42
)

TECHO_ALTO = 3000
modelo_M_alto = XGBClassifier(
    n_estimators=TECHO_ALTO, max_depth=DEPTH_ELEGIDO, learning_rate=LR_ELEGIDO,
    scale_pos_weight=(y_fit_final2 == 0).sum() / max((y_fit_final2 == 1).sum(), 1),
    eval_metric="aucpr", early_stopping_rounds=30, random_state=42,
)
modelo_M_alto.fit(X_fit_final2, y_fit_final2, eval_set=[(X_es_final2, y_es_final2)], verbose=False)
M_ALTO = modelo_M_alto.best_iteration + 1

convergio = M_ALTO < TECHO_ALTO * 0.95
print(f"Mejor combinacion: k={MEJOR_K}, fuente={MEJOR_FUENTE}")
print(f"M elegido con techo alto (n_estimators_max={TECHO_ALTO}, paciencia=30): {M_ALTO} "
      f"{'(convergio genuinamente, no llego al techo)' if convergio else '(todavia cerca del techo, considerar subirlo aun mas)'}")

## 9. (Opcional, lento) Verificar por BLAST remoto los candidatos con mayor probabilidad

Reutiliza `blast_remoto()` del cuaderno 03. **Cada consulta puede tardar 1-5 minutos** y NCBI limita la
tasa de consultas remotas -- por eso solo verificamos un puñado de los candidatos con mayor probabilidad,
no toda la lista. Un "sin hits" no descarta que sea un ARG real (podria ser genuinamente nuevo); un hit
contra un gen conocido no relacionado con resistencia sugiere que fue una falsa alarma del modelo.

In [10]:
from Bio.Blast import NCBIWWW, NCBIXML
import time


def blast_remoto(fasta_str, program="blastn", db="nt", hitlist=1, max_reintentos=3):
    ultimo = None
    for intento in range(1, max_reintentos + 1):
        try:
            handle = NCBIWWW.qblast(program, db, fasta_str, hitlist_size=hitlist, megablast=True)
            return list(NCBIXML.parse(handle))
        except Exception as e:
            ultimo = e
            print(f"  Intento {intento}/{max_reintentos} -> {type(e).__name__}: {e}")
            if intento < max_reintentos:
                time.sleep(20)
    raise RuntimeError(f"BLAST fallo tras {max_reintentos} intentos") from ultimo


N_VERIFICAR = 5
for _, fila in candidatos_nuevos.head(N_VERIFICAR).iterrows():
    print(f"\n--- {fila['orf_id']} (run={fila['run']}, proba={fila['proba_arg']:.2f}) ---")
    try:
        registros = blast_remoto(f">{fila['orf_id']}\n{fila['secuencia']}\n")
        rec = registros[0]
        if rec.alignments:
            top = rec.alignments[0]
            print(f"  Mejor hit: {top.hit_def[:90]}")
        else:
            print("  Sin hits en BLAST (posible secuencia genuinamente nueva, o ruido del modelo)")
    except Exception as e:
        print(f"  Fallo BLAST: {e}")


--- k119_3676_1 # 1 # 327 # 1 # ID=4095_1;partial=11;start_type=Edge;rbs_motif=None;rbs_spacer=None;gc_cont=0.667 (run=ERR1135451, proba=1.00) ---


  Mejor hit: Salmonella enterica subsp. enterica serovar Kentucky strain D2054 chromosome, complete gen

--- k119_1138_1 # 1 # 135 # -1 # ID=1133_1;partial=10;start_type=ATG;rbs_motif=GGAGG;rbs_spacer=5-10bp;gc_cont=0.689 (run=ERR1135438, proba=1.00) ---


  Mejor hit: MAG: Clostridiales bacterium isolate KR001_HIC_0007 chromosome, complete genome

--- k99_4625_2 # 603 # 677 # -1 # ID=7752_2;partial=01;start_type=Edge;rbs_motif=None;rbs_spacer=None;gc_cont=0.600 (run=ERR1135196, proba=1.00) ---


  Mejor hit: MAG: uncultured Alistipes sp. isolate metabat2_pooled_hifiasm.969 MAG genome assembly, chr

--- k119_6691_2 # 220 # 351 # -1 # ID=7867_2;partial=01;start_type=Edge;rbs_motif=None;rbs_spacer=None;gc_cont=0.606 (run=ERR1135463, proba=1.00) ---


  Sin hits en BLAST (posible secuencia genuinamente nueva, o ruido del modelo)

--- k99_3827_7 # 4242 # 4784 # 1 # ID=11941_7;partial=01;start_type=ATG;rbs_motif=GGAGG;rbs_spacer=5-10bp;gc_cont=0.523 (run=ERR1135222, proba=1.00) ---


  Sin hits en BLAST (posible secuencia genuinamente nueva, o ruido del modelo)


## 10. Discusion y limitaciones

- **Escala real todavia chica**: solo 12 muestras procesadas y 180 ARG confirmados por RGI para evaluar
  contra el mundo real. Suficiente para un primer piloto, no para conclusiones fuertes; correr el
  cuaderno 03 sobre mas `RUNS` de ENA daria un conjunto de evaluacion mas confiable.
- **Cambio de dominio CARD vs. metagenoma**: mitigamos la diferencia de longitud fragmentando, pero
  restan otras diferencias (calidad de ensamblaje, errores de secuenciacion, ORFs parciales en el borde
  de un contig) que CARD no tiene. Si el recall sobre hits conocidos es bajo, esta es la primera
  sospechosa.
- **Los "candidatos nuevos" no estan verificados**: sin una referencia externa (BLAST, revision experta,
  o idealmente validacion de laboratorio) no podemos afirmar que sean ARG reales, solo que el modelo los
  considera composicionalmente parecidos a los de CARD.
- **Si este baseline rinde mal**: siguientes pasos razonables, en orden de esfuerzo creciente, son
  probar k=5/6, features a nivel de proteina (k-mers de aminoacidos sobre `genes.faa`) en vez de ADN, y
  como ultimo recurso una CNN pequena sobre la secuencia one-hot -- pero dado que el entorno ya trae
  `xgboost`/`imbalanced-learn`/`kmc`, ese no deberia ser el primer lugar al que saltar.